In [ ]:
# Post training Stage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# we have prepared the data to train an RL algorithm for interleaved reasoning
# Stages


# Stage 1: Define Policy Model
# Stage 2: Build SFT pipeline to imitate think sub answer switch paradigm
# Stage 3: Build PPO algorithm and PPO loss
# Stage 4:

In [ ]:
# Install/Upgrade bitsandbytes for 8-bit quantization


!pip install -U bitsandbytes

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, BitsAndBytesConfig
import torch
import re


# This will fit:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)


# Define BitsAndBytesConfig for 8-bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# Load in 8-bit with torch_dtype=torch.float16 to avoid casting warnings
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.float16
)

# Explicitly set output_hidden_states to True in the model's config
model.config.output_hidden_states = True

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
# Add a new pad token and resize model embeddings correctly

tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model.resize_token_embeddings(len(tokenizer))

# Ensure the model config reflects the new pad token id
model.config.pad_token_id = tokenizer.pad_token_id
print(f'New pad token: {tokenizer.pad_token}, ID: {tokenizer.pad_token_id}')

New pad token: [PAD], ID: 151665


In [ ]:
tokens_sub_ans = tokenizer.tokenize('</sub_ans>')
tokens_final_ans = tokenizer.tokenize('</final_answer>')

print(f"'</sub_ans>' tokens: {tokens_sub_ans}")
print(f"'</sub_ans>' IDs: {tokenizer.convert_tokens_to_ids(tokens_sub_ans)}")

print(f"\n'</final_answer>' tokens: {tokens_final_ans}")
print(f"'</final_answer>' IDs: {tokenizer.convert_tokens_to_ids(tokens_final_ans)}")

'</sub_ans>' tokens: ['</', 'sub', '_ans', '>']
'</sub_ans>' IDs: [522, 1966, 60801, 29]

'</final_answer>' tokens: ['</', 'final', '_answer', '>']
'</final_answer>' IDs: [522, 11822, 28534, 29]


In [ ]:
# Lora technique

from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=32,  # LoRA rank
                        lora_alpha=16,  # Scaling factor
                        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],  # Target layers for Qwen2.5
                        bias="none",
                        task_type="CAUSAL_LM")

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Check trainable parameters

trainable params: 80,740,352 || all params: 7,693,504,000 || trainable%: 1.0495


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"Total Memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("No GPU available.")

GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total Memory: 101.97 GB


In [ ]:
tokenizer.model_max_length

131072

In [ ]:
print(tokenizer.eos_token, tokenizer.pad_token)

<|im_end|> [PAD]


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Data/interleaved_reasoning_datatset_210_tasks.csv")
df.head()

,task_id,reasoning_step,sub_answer,final_answer,quiz_text,row_claims
0,task_10000,<think>We find that Elijah being a guardian is...,<answer>Assuming Elijah is a guardian forces t...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
1,task_10000,<think>Let Elijah be a deceiver. The false cla...,"<answer>Since deceivers always lie, Elijah bei...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
2,task_10000,<think>Ella as a guardian is ruled out: the tr...,<answer>Assuming Ella is a guardian and that S...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
3,task_10000,<think>Assume Ella is a deceiver. Their lie ab...,<answer>Assuming Ella is a deceiver does not l...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
4,task_10000,<think>Let us suppose Scarlett is a guardian. ...,"<answer>With Scarlett as a guardian, the state...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Scarlett"", ""role"": ""guardian""}]"


In [ ]:
prompts = list(df['quiz_text'].unique())
len(prompts)

196

In [ ]:
# final_df = df.groupby('task').agg(think_steps=('gt_think_steps', lambda x: x.dropna().tolist()),
#                                   answers=('gt_answers', lambda x: x.dropna().tolist()),
#                                   final_answer=('gt_final_answer', lambda x: x.dropna().tolist())).reset_index(drop=False)


In [ ]:


# final_df.head()

In [ ]:
## ppo algorithm

# roll outs

In [ ]:
EXAMPLES = """User: A very special island is inhabited only by wizards and goblins. Wizards always tell the truth, and goblins always lie. You meet 2 inhabitants: Quinn and Rowan. Quinn said that Rowan is a goblin. Rowan said "Quinn is a wizard and Rowan is a wizard". So who is a wizard and who is a goblin?. Assistant: <think>Suppose Quinn is a goblin. Then Quinn's statement is false, so Rowan is a wizard. A wizard tells the truth, but Rowan's statement says Quinn is a wizard, which would be false. That is a contradiction, so Quinn is a wizard.</think>
<answer>Quinn is a wizard
<sub_ans>Quinn: wizard</sub_ans></answer>
<think>Quinn is a wizard, so Quinn's statement is true, and Rowan is a goblin. Check: Rowan's statement says Rowan is a wizard, which is false, and a goblin always lies, so this fits.</think>
<answer>Rowan is a goblin
<sub_ans>Rowan: goblin</sub_ans></answer>
<think>So, the final answer is:</think>
<answer>Quinn is a wizard, and Rowan is a goblin.
<final_answer>{"Quinn": "wizard", "Rowan": "goblin"}</final_answer></answer>
User: A very special island is inhabited only by sailors and pirates. Sailors always tell the truth, and pirates always lie. You meet 3 inhabitants: Tessa, Milo, and Iris. Tessa said "Tessa is a sailor or Tessa is a pirate". Milo said that Tessa is a pirate. Iris said "Milo is a pirate if and only if Tessa is a sailor". So who is a sailor and who is a pirate?. Assistant: <think>Tessa says "Tessa is a sailor or Tessa is a pirate". Everyone is either a sailor or a pirate, so this statement is always true. A pirate never says something true, so Tessa is a sailor.</think>
<answer>Tessa is a sailor
<sub_ans>Tessa: sailor</sub_ans></answer>
<think>Milo says Tessa is a pirate. Tessa is a sailor, so this statement is false. A sailor never says something false, so Milo is a pirate.</think>
<answer>Milo is a pirate
<sub_ans>Milo: pirate</sub_ans></answer>
<think>Iris says "Milo is a pirate if and only if Tessa is a sailor". Milo is a pirate and Tessa is a sailor, so both sides are true and the statement is true. A pirate never says something true, so Iris is a sailor.</think>
<answer>Iris is a sailor
<sub_ans>Iris: sailor</sub_ans></answer>
<think>So, the final answer is:</think>
<answer>Tessa is a sailor, Milo is a pirate, and Iris is a sailor.
<final_answer>{"Tessa": "sailor", "Milo": "pirate", "Iris": "sailor"}</final_answer></answer>
"""


TABLE1 = """You are a helpful assistant. You reason through problems step by step before providing an answer. You
conduct your reasoning within <think></think> and share partial answers within <answer></answer>
as soon as you become confident about the intermediate results. You continue this pattern of
<think></think><answer></answer> until you reach the final answer. User: prompt. Assistant:
"""

SUB_ANS_SENTENCE = ("Write one <think></think> before every <answer></answer>, and never write two <answer></answer> blocks in a row. "
                    "Each <answer></answer> states one character's role in a short sentence, followed on the next line by the same role as a tag "
                    "inside the same <answer></answer>, using the names and role names from the puzzle. "
                    "For example, for a character named Tom who is not in this puzzle: "
                    "<think>Tom's statement is true, so Tom tells the truth.</think><answer>Tom is {a_truth}\n<sub_ans>Tom: {truth}</sub_ans></answer>")

FINAL_SENTENCE = ("The last <answer></answer> also comes after a <think></think>. It states the final answer, followed on the next line by "
                  "the full assignment of the puzzle's characters as a JSON object inside <final_answer></final_answer>.")

def build_template(table1):
    text = " ".join(table1.split())
    assert "PASTE TABLE 1" not in text, "Paste Table 1 of the paper into TABLE1 first."
    cut = text.find("User:")
    sentences = re.split(r"(?<=\.)\s+", text[:cut].strip())
    assert cut > 0 and len(sentences) == 4, "Table 1 should have 4 instruction sentences, then 'User:'."
    return " ".join(sentences[:3] + [SUB_ANS_SENTENCE, sentences[3], FINAL_SENTENCE]) + " " + EXAMPLES + \
           text[cut:].replace("prompt", "{question}", 1)

TEMPLATE = build_template(TABLE1)
TEMPERATURE = 0.8

In [ ]:
def generate_input(problem, tokenizer):
    """
    Generate reasoning with SHORT think-answer pairs (1-2 sentences each)
    """

    truth, liar = re.search(r"inhabited only by (\w+) and (\w+)\.", problem).groups()
    truth, liar = truth[:-1], liar[:-1]
    a_truth = ("an " if truth[0] in "aeiou" else "a ") + truth
    prompt = (TEMPLATE.replace("{a_truth}", a_truth).replace("{truth}", truth).replace("{liar}", liar)
                      .replace("{question}", problem))

    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    return enc["input_ids"]

In [ ]:
df.columns

Index(['task_id', 'reasoning_step', 'sub_answer', 'final_answer', 'quiz_text',
       'row_claims'],
      dtype='object')

In [ ]:
df.head()

,task_id,reasoning_step,sub_answer,final_answer,quiz_text,row_claims
0,task_10000,<think>We find that Elijah being a guardian is...,<answer>Assuming Elijah is a guardian forces t...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
1,task_10000,<think>Let Elijah be a deceiver. The false cla...,"<answer>Since deceivers always lie, Elijah bei...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
2,task_10000,<think>Ella as a guardian is ruled out: the tr...,<answer>Assuming Ella is a guardian and that S...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
3,task_10000,<think>Assume Ella is a deceiver. Their lie ab...,<answer>Assuming Ella is a deceiver does not l...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
4,task_10000,<think>Let us suppose Scarlett is a guardian. ...,"<answer>With Scarlett as a guardian, the state...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Scarlett"", ""role"": ""guardian""}]"


In [ ]:
final_df = df.groupby(['task_id', 'quiz_text']).agg(think_steps=('reasoning_step', lambda x: x.dropna().tolist()),
                                  answers=('sub_answer', lambda x: x.dropna().tolist()),
                                  row_claims=('row_claims', lambda x: x.dropna().tolist()),
                                  final_answer=('final_answer', lambda x: x.dropna().tolist()[0])).reset_index(drop=False)

final_df.head()

,task_id,quiz_text,think_steps,answers,row_claims,final_answer
0,task_10000,A very special island is inhabited only by gua...,[<think>We find that Elijah being a guardian i...,[<answer>Assuming Elijah is a guardian forces ...,"[[{""name"": ""Elijah"", ""role"": ""deceiver""}], [{""...",<final>Elijah is a deceiver and Ella is a dece...
1,task_1001491,A very special island is inhabited only by pio...,[<think>Assuming Evelyn is a pioneer creates a...,[<answer>Assuming Evelyn is a pioneer forces t...,"[[{""name"": ""Evelyn"", ""role"": ""laggard""}], [{""n...",<final>Evelyn is a laggard and Jack is a pione...
2,task_1006812,A very special island is inhabited only by sai...,[<think>Assume Joseph is a saint. This leads t...,[<answer>The logical chain from Joseph being a...,"[[{""name"": ""Joseph"", ""role"": ""sinner""}], [{""na...",<final>Joseph is a sinner and Ethan is a saint...
3,task_100831,A very special island is inhabited only by sai...,[<think>Suppose Luke is a saint. Then Sebastia...,[<answer>We derive a contradiction from Luke b...,"[[{""name"": ""Luke"", ""role"": ""sinner""}], [{""name...",<final>Luke is a sinner and Sebastian is a sai...
4,task_1011056,A very special island is inhabited only by alt...,[<think>Assume Isabella is an altruist. This l...,[<answer>We derive a contradiction from Isabel...,"[[{""name"": ""Isabella"", ""role"": ""egoist""}], [{""...",<final>Isabella is an egoist and Carter is an ...


In [ ]:
final_df['answers'][0]

['<answer>Assuming Elijah is a guardian forces the claim Scarlett is a deceiver to be true, but this creates an impossible situation. Elijah is a deceiver.</answer>',
 '<answer>Since deceivers always lie, Elijah being a deceiver means the claim Scarlett is a deceiver is false. This falsity creates no issues.</answer>',
 '<answer>Assuming Ella is a guardian and that Scarlett is a deceiver and Ella is a deceiver is truthful produces irreconcilable conditions. Therefore, Ella must be a deceiver.</answer>',
 '<answer>Assuming Ella is a deceiver does not lead to any contradiction, as their statement being false is consistent with deceivers always lying.</answer>',
 '<answer>With Scarlett as a guardian, the statement Elijah is a deceiver is taken as fact. No contradiction emerges from this assignment.</answer>',
 '<answer>The denial of Elijah is a deceiver under the deceiver hypothesis for Scarlett creates a chain of impossible deductions. Scarlett cannot be a deceiver.</answer>',
 '<answer>

In [ ]:
class StopAtFinalAnswer(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, input_ids, scores, **kwargs):
        decoded = self.tokenizer.decode(input_ids[0][-20:], skip_special_tokens=True)
        return "</final_answer>" in decoded

In [ ]:
# from torch import nn

# class PPO_Model(nn.Module):

#   def __init__(self, lm):
#     super().__init__()
#     self.lm = lm
#     self.value_head = nn.Linear(lm.config.hidden_size, 1)

#   def forward(self, input_ids, past_key_values=None):
#     # Pass past_key_values and set use_cache=True for efficient autoregressive decoding
#     outputs = self.lm(input_ids, output_hidden_states=True, past_key_values=past_key_values, use_cache=True)

#     logits = outputs.logits
#     # Ensure last_hidden is on the same device as value_head AND cast to float32
#     # outputs.hidden_states[-1] will be (batch_size, sequence_length, hidden_size)
#     # If only the last token is passed as input_ids (with past_key_values), sequence_length will be 1.
#     last_hidden = outputs.hidden_states[-1][:, -1]
#     last_hidden = last_hidden.to(self.value_head.weight.device).float()

#     value = self.value_head(last_hidden).squeeze(-1)
#     return logits, value, outputs.past_key_values # Return updated past_key_values


from torch import nn

class PPO_Model(nn.Module):

  def __init__(self, lm):
    super().__init__()
    self.lm = lm
    self.value_head = nn.Linear(lm.config.hidden_size, 1)

  def forward(self, input_ids, past_key_values, use_cache):
    # Pass past_key_values and set use_cache=True for efficient autoregressive decoding
    outputs = self.lm(input_ids, output_hidden_states=True, past_key_values=past_key_values, use_cache=use_cache)

    logits = outputs.logits
    # Ensure last_hidden is on the same device as value_head AND cast to float32
    # outputs.hidden_states[-1] will be (batch_size, sequence_length, hidden_size)
    # If only the last token is passed as input_ids (with past_key_values), sequence_length will be 1.

    if use_cache == True:
      last_hidden = outputs.hidden_states[-1][:, -1, :]
      last_hidden = last_hidden.to(self.value_head.weight.device).float()
      value = self.value_head(last_hidden).squeeze(-1)

      return logits, value, outputs.past_key_values # Return updated past_key_values

    else:
      last_hidden = outputs.hidden_states[-1]
      last_hidden = last_hidden.to(self.value_head.weight.device).float()
      value = self.value_head(last_hidden).squeeze(-1)

      return logits, value



In [ ]:
model.config.hidden_size

3584

In [ ]:
torch.tensor([1,2,3]).unsqueeze(-1)

tensor([[1],
        [2],
        [3]])

In [ ]:
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"


policy_model = PPO_Model(model)
# ref_model = copy.deepcopy(policy_model)

# ref_model.eval()          # important
# for p in ref_model.parameters():
#     p.requires_grad = False

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
policy_model.to(device)
# ref_model.to(device)

PPO_Model(
  (lm): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): Qwen2ForCausalLM(
        (model): Qwen2Model(
          (embed_tokens): Embedding(151666, 3584)
          (layers): ModuleList(
            (0-27): 28 x Qwen2DecoderLayer(
              (self_attn): Qwen2Attention(
                (q_proj): lora.Linear8bitLt(
                  (base_layer): Linear8bitLt(in_features=3584, out_features=3584, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=3584, out_features=32, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=32, out_features=3584, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()

In [ ]:
final_df['task_id'].nunique()

196

In [ ]:
# Check if gradient checkpointing is enabled via the proper attribute
print(f"Gradient Checkpointing Enabled: {policy_model.lm.is_gradient_checkpointing}")

Gradient Checkpointing Enabled: True


In [ ]:
def extract_format(generated_subanswers, name_role_subanswers_gen):
    answers = [a.strip() for a in generated_subanswers]
    if len(answers) < 2:
        return -1

    *steps, last = answers

    for a in steps:
        tags = re.findall(r"<sub_ans>(.*?)</sub_ans>", a, re.S)
        sentence = re.sub(r"<sub_ans>.*?</sub_ans>", "", a, flags=re.S).strip()
        if len(tags) != 1 or not re.fullmatch(r"\s*\w+\s*:\s*\w+\s*", tags[0]) or not sentence:
            return -1   # each step answer needs a sentence and exactly one tag holding one "Name: role" pair
    if "<sub_ans>" in last or not re.search(r"<final_answer>.*</final_answer>\s*$", last, re.S):
        return -1       # the last answer ends with the final answer and holds no tag

    if len(name_role_subanswers_gen) != len(steps):
        return -1
    return 1

In [ ]:
###
# build time discounted reward. according paper rbase

In [ ]:
import json
import re

def extract_final(final_generated_answer, gt_final_pair):

  try:
    final_generated_answer = json.loads(final_generated_answer)

  except (json.JSONDecodeError, TypeError):
    return False

  # must be a non-empty dict of str -> str; anything else is a malformed answer, not a crash
  if not (isinstance(final_generated_answer, dict) and final_generated_answer
          and all(isinstance(k, str) and isinstance(v, str) for k, v in final_generated_answer.items())):
    return False

  print('gt_final_pair_new: ', gt_final_pair)

  print('final_generated_answer: ', final_generated_answer)

# gt_final_pair_new:  ['[{"name": "Zoey", "role": "devil"}]', '[{"name": "Zoey", "role": "devil"}]', '[{"name": "Audrey", "role": "angel"}]', '[{"name": "Audrey", "role": "angel"}]', '[{"name": "Zoey", "role": "devil"}, {"name": "Audrey", "role": "angel"}]']
# final_generated_answer:  {'Zoey': 'devil', 'Audrey': 'angel'}

  try:
    gt_final_pair = json.loads(gt_final_pair)  # last ROW, all entities

  except (json.JSONDecodeError, TypeError, IndexError):
      return False

  if not gt_final_pair or not final_generated_answer:
      return False

  gt_final = {pair["name"].lower() : pair["role"].lower() for pair in list(gt_final_pair)}

  gen_final = {key.lower(): value.lower() for key, value in final_generated_answer.items()}

  print(f'ground truth final naswer: {gt_final} and generated final answer {gen_final}')


  gt_keys = list(sorted(gt_final.keys()))
  gen_keys = list(sorted(gen_final.keys()))



  if gt_keys == gen_keys:
    for key in gt_keys:
      if gen_final[key] != gt_final[key]:
        return False

  else:
    return False

  return True

In [ ]:
def norm(s):
    name, _, role = s.partition(':')
    return f"{name.strip().lower()}:{role.strip().lower()}"

In [ ]:
def conditional_reward_system(gt_clean_subanswers, generated_subanswers, name_role_subanswers_gen, name_role_subanswers_gt,
                              final_generated_answer, final_answer, gt_final_pair, sub_ans_token_positions, final_ans_index,
                              Acc_curr, Acc_prev, mode, T, gamma):

    rewards = torch.zeros(T, device=device)

    R_base = 0.5
    epsilon = 0.05

    matched = []  # sub_ans -> step index


    r_format = extract_format(generated_subanswers, name_role_subanswers_gen)


    if r_format != 1:
        r_final = 0                      # paper: final reward only when the format is correct
    elif extract_final(final_generated_answer, gt_final_pair):
        r_final = 2
    else:
        try:
            json.loads(final_generated_answer)
            r_final = -1.5               # paper: wrong answer
        except (json.JSONDecodeError, TypeError):
            r_final = -2                 # paper: answer cannot be parsed

    bool_format = 1 if r_format==1 else 0

    bool_final = 1 if r_final>0 else 0


    matches=0

    len_subans_gt = len(name_role_subanswers_gt)



    if len_subans_gt == 0:
        # place format + final first, unconditionally
        rewards[final_ans_index] += r_final
        rewards[T - 1]           += r_format

        return rewards

    # Parse JSON strings and dedupe to unique claims
    parsed_gt = set()
    for pair in name_role_subanswers_gt:
        for d in json.loads(pair):
            parsed_gt.add(f"{d['name'].strip().lower()}:{d['role'].strip().lower()}")

    gt_pairs       = list(parsed_gt)             # unique claims, lowercased
    gen_pairs      = [norm(p) for p in name_role_subanswers_gen]
    len_subans_gt  = len(gt_pairs)


    for k, (gen_pair, t_pos) in enumerate(zip(gen_pairs, sub_ans_token_positions), start=1):
        if gen_pair in gt_pairs:
            gt_pairs.remove(gen_pair)
            matches += 1
            matched.append((k, t_pos))

    # current_accuracy = matches / len_subans_gt if len_subans_gt > 0 else 0.0

    # bool_accuracy = current_accuracy > prev_acc- epsilon



    if mode == 'training_phase':
      bool_accuracy = 1 if (Acc_curr > Acc_prev - epsilon) else 0

    else:
      bool_accuracy = 1

    gate = bool_format * bool_final * bool_accuracy


    ### calc reward (time discounted)


    matched_pos = []
    matched_int_rewards = []

    for k, t_pos in matched:
      if matches==len_subans_gt:
        r_intermediate = gate * R_base / matches
      else:
        r_intermediate  = gate * R_base * (1.0 / k) / len_subans_gt
      rewards[t_pos]+= r_intermediate
      matched_pos.append(t_pos)
      matched_int_rewards.append(r_intermediate)



    # (2) final-answer reward at the </final_answer> token

    final_ans_token_pos = final_ans_index
    rewards[final_ans_token_pos] += r_final


    rewards[T-1]+= r_format

    print(f'rewards for the task in rollout {i+1}: ',  rewards)

    print(f'indexes to watch for format, final_nas_token_pos and matched_pos in rewards: {T-1, final_ans_token_pos, matched_pos}')

    print(f'r_final: {r_final}, r_format: {r_format}, matched_intermediate_rewards: {matched_int_rewards}')

    print('\n')

    return rewards

In [ ]:
final_df.head()

,task_id,quiz_text,think_steps,answers,row_claims,final_answer
0,task_10000,A very special island is inhabited only by gua...,[<think>We find that Elijah being a guardian i...,[<answer>Assuming Elijah is a guardian forces ...,"[[{""name"": ""Elijah"", ""role"": ""deceiver""}], [{""...",<final>Elijah is a deceiver and Ella is a dece...
1,task_1001491,A very special island is inhabited only by pio...,[<think>Assuming Evelyn is a pioneer creates a...,[<answer>Assuming Evelyn is a pioneer forces t...,"[[{""name"": ""Evelyn"", ""role"": ""laggard""}], [{""n...",<final>Evelyn is a laggard and Jack is a pione...
2,task_1006812,A very special island is inhabited only by sai...,[<think>Assume Joseph is a saint. This leads t...,[<answer>The logical chain from Joseph being a...,"[[{""name"": ""Joseph"", ""role"": ""sinner""}], [{""na...",<final>Joseph is a sinner and Ethan is a saint...
3,task_100831,A very special island is inhabited only by sai...,[<think>Suppose Luke is a saint. Then Sebastia...,[<answer>We derive a contradiction from Luke b...,"[[{""name"": ""Luke"", ""role"": ""sinner""}], [{""name...",<final>Luke is a sinner and Sebastian is a sai...
4,task_1011056,A very special island is inhabited only by alt...,[<think>Assume Isabella is an altruist. This l...,[<answer>We derive a contradiction from Isabel...,"[[{""name"": ""Isabella"", ""role"": ""egoist""}], [{""...",<final>Isabella is an egoist and Carter is an ...


In [ ]:
df.head()

,task_id,reasoning_step,sub_answer,final_answer,quiz_text,row_claims
0,task_10000,<think>We find that Elijah being a guardian is...,<answer>Assuming Elijah is a guardian forces t...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
1,task_10000,<think>Let Elijah be a deceiver. The false cla...,"<answer>Since deceivers always lie, Elijah bei...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Elijah"", ""role"": ""deceiver""}]"
2,task_10000,<think>Ella as a guardian is ruled out: the tr...,<answer>Assuming Ella is a guardian and that S...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
3,task_10000,<think>Assume Ella is a deceiver. Their lie ab...,<answer>Assuming Ella is a deceiver does not l...,NaN,A very special island is inhabited only by gua...,"[{""name"": ""Ella"", ""role"": ""deceiver""}]"
4,task_10000,<think>Let us suppose Scarlett is a guardian. ...,"<answer>With Scarlett as a guardian, the state...",NaN,A very special island is inhabited only by gua...,"[{""name"": ""Scarlett"", ""role"": ""guardian""}]"


In [ ]:
AMBIGUOUS_TASKS = ["task_1017871", "task_1034461", "task_119605", "task_142859", "task_225829",
                   "task_271847", "task_358749", "task_370114", "task_41478", "task_434129",
                   "task_460836", "task_544449", "task_557160", "task_563502", "task_693774",
                   "task_810287", "task_810932", "task_812611", "task_822063", "task_839787",
                   "task_858604", "task_900611", "task_931103", "task_949585", "task_991505"]
final_df = final_df[~final_df['task_id'].isin(AMBIGUOUS_TASKS)]
# final_df = final_df.head(5)


In [ ]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 171 entries, 0 to 194
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   task_id       171 non-null    object
 1   quiz_text     171 non-null    object
 2   think_steps   171 non-null    object
 3   answers       171 non-null    object
 4   row_claims    171 non-null    object
 5   final_answer  171 non-null    object
dtypes: object(6)
memory usage: 9.4+ KB


In [ ]:
import pickle
import os

trajectories_file = '/content/drive/MyDrive/trajectories_paper_template_v2.pkl'

if os.path.exists(trajectories_file):
    with open(trajectories_file, 'rb') as f:
        trajectories = pickle.load(f)
    print(f"Loaded {len(trajectories)} trajectories from {trajectories_file}")
else:
    trajectories = {}
    print("No existing trajectories found. Starting fresh.")

No existing trajectories found. Starting fresh.


In [ ]:
final_df[final_df['task_id']=='task_10000']['row_claims'].dropna().to_list()[0][-1]

'[{"name": "Elijah", "role": "deceiver"}, {"name": "Ella", "role": "deceiver"}, {"name": "Scarlett", "role": "guardian"}]'

In [ ]:
import torch.nn.functional as F
import logging
import warnings

# Suppress bitsandbytes warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="bitsandbytes")

# ref_model = PPO_Model(model) # This line was removed as ref_model is already defined and moved to device

policy_model.eval()   # in train mode, gradient checkpointing switches off the cache this loop relies on
gamma = 0.999
k=4

# Extract already processed task IDs for fast O(1) lookup
processed_task_ids = {k[0] for k in trajectories.keys()}

task_id_list, prompts_list, final_answers_list_gt, row_claims = final_df['task_id'].to_list(), final_df['quiz_text'].to_list(), final_df['final_answer'].to_list(), final_df['row_claims'].to_list()

gt_final_answers = [claims[-1] for claims in row_claims]
print('final_answer gt:', gt_final_answers)

for task_id, input_text, final_answer_gt, gt_final_pair in zip(task_id_list, prompts_list, final_answers_list_gt, gt_final_answers):

  # Skip if already processed
  if task_id in processed_task_ids:
    print(f'skipping the current task {task_id} because its already processed')
    continue

  MAX_RETRIES = 4

  print('\n')
  print(f'task_id: {task_id}')
  print('\n')

  match_val = re.search(r"<final>(.*?)</final>", final_answer_gt, re.S)
  final_answer_gt = match_val.group(1).strip() if match_val is not None else ''

  for i in range(k):

    gt_subanswers = df[df['task_id']==task_id]['sub_answer'].dropna().to_list()
    name_role_subanswers_gt = df[df['task_id']==task_id]['row_claims'].dropna().to_list()

    print(f'for rollout {i+1}')
    print('\n')

    all_retry = True

    for retry in range(1, MAX_RETRIES+1):
      # Generate initial prompt input_ids once per retry
      initial_prompt_input_ids = generate_input(input_text, tokenizer)

      actions, log_probs_old, values = [], [], []
      past_key_values = None # Initialize past_key_values for efficient generation
      current_model_input_ids = initial_prompt_input_ids # First input to the model will be the full prompt

      if retry == MAX_RETRIES:
        print('all retries_failed, directing for another trajectory')
        all_retry = False
        break

      # Loop for generating tokens - INCREASED LIMIT HERE
      for _ in range(600):
        with torch.no_grad():
            # Pass current_model_input_ids and past_key_values to the model
            logits, value, past_key_values = policy_model(current_model_input_ids, past_key_values, True)

        # Logits for the *last* token in the current_model_input_ids
        probs = F.softmax(logits[:, -1, :] / TEMPERATURE, dim=-1)

        dist = torch.distributions.Categorical(probs)

        action = dist.sample()                           # sampling a token_id

        log_prob = dist.log_prob(action)

        actions.append(action) # Collect the generated token
        log_probs_old.append(log_prob.detach()) # Store log_prob of the sampled action
        values.append(value.detach()) # Store value prediction for the state before action

        # For the next step, the input to the model is only the newly generated token
        current_model_input_ids = action.unsqueeze(0) # Prepare for the next iteration

        # Check for final answer token to stop generation early
        # We need to concatenate initial_prompt_input_ids with all generated actions for the stopping criteria
        # This temporary tensor is only for checking, not for model input
        temp_full_sequence_ids = torch.cat([initial_prompt_input_ids.squeeze(0), torch.cat(actions)], dim=0).unsqueeze(0)
        if StopAtFinalAnswer(tokenizer)(temp_full_sequence_ids, None):
            break


      # Decode all collected actions to get the full generated text
      generated_text = tokenizer.decode(
          torch.cat(actions),
          skip_special_tokens= True
      )

      end = generated_text.rfind("</final_answer>")
      start = generated_text.rfind("<final_answer>", 0, max(end, 0))
      if end != -1 and start != -1:
          generated_text = generated_text[:end + len("</final_answer>")]   # drop anything after the tag, like the "</" of a "></" token
          before = generated_text[:start]
          if before.count("<answer>") == before.count("</answer>"):   # final answer written outside any <answer>
              generated_text = before + "<answer>" + generated_text[start:]
          if generated_text.count("<answer>") > generated_text.count("</answer>"):
              generated_text = generated_text + "</answer>"

      think = re.findall(r"<think>(.*?)</think>", generated_text, re.S)
      generated_subanswers = [a for a in re.findall(r"<answer>(.*?)</answer>", generated_text, re.S) if a.strip()]
      name_role_subanswers_generated = re.findall(r"<sub_ans>(.*?)</sub_ans>", generated_text, re.S)

      # Clean the ground truth subanswers to extract text from tags if they exist
      gt_clean_subanswers = []
      for sa in gt_subanswers:
          if not pd.isna(sa):
              match = re.search(r"<answer>(.*?)</answer>", str(sa), re.S)
              gt_clean_subanswers.append(match.group(1).strip() if match else str(sa).strip())

      # Fallback if ground truth is empty or missing
      if not gt_clean_subanswers:
          gt_clean_subanswers = [''] * len(generated_subanswers)



      match_val = re.search(r"<final_answer>(.*?)</final_answer>", generated_text, re.S)
      final_generated_answer = match_val.group(1).strip() if match_val is not None else ''


      if '<think>' not in generated_text or '</think>' not in generated_text:
        print(f'Attempting retry because think step is missing, retry no. {retry}')
        continue

      if '<answer>' not in generated_text or '</answer>' not in generated_text:
        print(f'Attempting retry because <answer> or </answer> step is missing, retry no. {retry}')
        continue

      if '<final_answer>' not in generated_text or '</final_answer>' not in generated_text:
        print(f'Attempting retry because <final_answer> or </final_answer> step is missing, retry no. {retry}')
        continue

      if '<sub_ans>' not in generated_text or '</sub_ans>' not in generated_text:
        print(f'Attempting retry because <sub_ans> or </sub_ans> step is missing, retry no. {retry}')
        continue

      action_ids = [a.item() for a in actions]
      sub_ans_indexes = []
      final_ans_index = []

      for m in range(len(action_ids)):
        text_so_far = tokenizer.decode(action_ids[:m+1], skip_special_tokens=True)
        prev_text = tokenizer.decode(action_ids[:m], skip_special_tokens=True) if m > 0 else ""

        # Use count() to robustly check if the tag was completed by token m
        if text_so_far.count("</sub_ans>") > prev_text.count("</sub_ans>"):
          sub_ans_indexes.append(m)

        if text_so_far.count("</final_answer>") > prev_text.count("</final_answer>"):
          final_ans_index.append(m)

      print('\nsub_ans_indexes: \n', sub_ans_indexes)
      print('\nfinal_ans_index: \n', final_ans_index)
      print('name_role_subanswers_generated: \n', name_role_subanswers_generated)

      if len(sub_ans_indexes) != len(name_role_subanswers_generated):
        print(f'Attempting retry because sub_ans_indexes do not match name_role_subanswers_generated, retry no. {retry}')
        continue

      if len(final_ans_index) != 1:
        print(f'Attempting retry because final_ans_index is not 1, retry no. {retry}')
        continue

      # print('think_steps: \n', think)
      print('name_role_subanswers_gt: \n', name_role_subanswers_gt)

      print('\ngenerated_subanswers: \n', generated_subanswers)
      print('final_generated_answer: \n', final_generated_answer)

      print('\ngt_clean_subanswers: \n', gt_clean_subanswers)

      print('\nfinal_answer_gt: \n', final_answer_gt)


      print('\n')
      print('all successful currently with retry or no retries')

      break

    if all_retry:
        print('reward calculation: ')

        T = len(values)
        # calculation reward/return for each task for advantage estimation i.e per trajectory
        rewards = conditional_reward_system(gt_clean_subanswers, generated_subanswers, name_role_subanswers_generated, name_role_subanswers_gt,
                                           final_generated_answer, final_answer_gt, gt_final_pair, sub_ans_indexes, final_ans_index[0],
                                           None, None, "no_accuracy_gate", T, gamma)

        # print(f'prev_accuracy: ', prev_accuracy)
        # print('\n')


        # Corrected: Initialize returns as (T, 1) to match values shape
        returns = torch.zeros(T, 1).to(device) # Move returns to the same device as values
        returns[T - 1] = rewards[T - 1]

        for t in reversed(range(T - 1)):
            returns[t] = rewards[t] + gamma * returns[t + 1]

        values  = torch.stack(values)

        advantages = (returns - values).detach()

        # pick the first task, do one rollout, verify shapes and values
        assert len(action_ids) == T
        assert len(sub_ans_indexes) == len(name_role_subanswers_generated)
        assert 0 <= final_ans_index[0] < T
        assert all(0 <= p < T for p in sub_ans_indexes)
        assert all(isinstance(p, str) for p in name_role_subanswers_gt)   # catches NaN
        assert -3.5 <= rewards.sum().item() <= 4.5
        assert returns[0].abs().item() > 0  # confirms terminal reward propagated


        trajectories[(task_id, i+1)] = {
            "actions": torch.stack(actions),
            "log_probs_old": torch.stack(log_probs_old),
            "advantages": advantages,
            "returns": returns,
            'rewards': rewards,
            'generated_subanswers': generated_subanswers,
            'name_role_subanswers_generated': name_role_subanswers_generated,
            'final_generated_answer': final_generated_answer,
            'sub_ans_token_positions': sub_ans_indexes,
            'final_ans_index': final_ans_index,
            'T': T,
            'generated_text': generated_text,
        }

        print('\n')

    # in the failure branch:
    else:
        print('all retries failed, skipping rollout')
        continue

  with open(trajectories_file, 'wb') as f:   # save after every task, so a disconnect loses at most one task
    pickle.dump(trajectories, f)


Streaming output truncated to the last 5000 lines.

final_answer_gt: 
 Charlotte is a pioneer and Camila is a pioneer.


all successful currently with retry or no retries
reward calculation: 
gt_final_pair_new:  [{"name": "Charlotte", "role": "pioneer"}, {"name": "Camila", "role": "pioneer"}]
final_generated_answer:  {'Charlotte': 'pioneer', 'Camila': 'pioneer'}
ground truth final naswer: {'charlotte': 'pioneer', 'camila': 'pioneer'} and generated final answer {'charlotte': 'pioneer', 'camila': 'pioneer'}
rewards for the task in rollout 1:  tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.000

In [ ]:
len(trajectories.keys())

684

In [ ]:
import pickle

with open(trajectories_file, 'wb') as f:
    pickle.dump(trajectories, f)

In [ ]:
with open("/content/rollouts_raw.jsonl", "w") as f:
    for (task_id, r), tr in trajectories.items():
        f.write(json.dumps({"task_id": task_id, "rollout": r, "text": tr["generated_text"]}) + "\n")

In [ ]:
# ref_model.to("cpu")

# import torch
# torch.cuda.empty_cache()

In [ ]:
print(torch.cuda.memory_allocated() / 1e9)
print(torch.cuda.memory_reserved() / 1e9)

11.363336704
13.551796224


In [ ]:
def ppo_loss(ratio, advantage, value_pred, return_val, entropy):
  eps = 0.2
  c_v = 0.5
  c_h = 0.01

  l_policy = -torch.min((ratio * advantage), (torch.clamp(ratio, 1 - eps, 1 + eps) * advantage))
  l_policy = l_policy.mean() # Reduce policy loss to scalar

  value_t = value_pred
  # Ensure return_val has the same shape as value_t (torch.Size([1]))
  if return_val.dim() == 0: # if it's a scalar
      return_val = return_val.unsqueeze(0) # make it torch.Size([1])

  l_value = F.mse_loss(value_t, return_val)


  return l_policy + c_v * l_value - c_h * entropy

In [ ]:
task_ids = []

for task_id in list(trajectories.keys()):
  if task_id[0] not in task_ids:
    task_ids.append(task_id[0])

print(task_ids)

['task_10000', 'task_1001491', 'task_1006812', 'task_100831', 'task_1011056', 'task_1011210', 'task_1013977', 'task_1015088', 'task_1018710', 'task_1025006', 'task_1025939', 'task_1029150', 'task_1033235', 'task_1035282', 'task_1043067', 'task_1048367', 'task_1055891', 'task_1064097', 'task_1070810', 'task_107264', 'task_109318', 'task_110615', 'task_119266', 'task_121903', 'task_123730', 'task_131118', 'task_140447', 'task_148091', 'task_153230', 'task_15745', 'task_159016', 'task_167558', 'task_174595', 'task_175932', 'task_184592', 'task_184773', 'task_194608', 'task_195811', 'task_20392', 'task_204524', 'task_209865', 'task_214851', 'task_216665', 'task_217082', 'task_231774', 'task_240680', 'task_243811', 'task_247070', 'task_256179', 'task_260823', 'task_266524', 'task_281767', 'task_282309', 'task_282829', 'task_284382', 'task_286334', 'task_290254', 'task_297166', 'task_29949', 'task_305791', 'task_314641', 'task_319683', 'task_323643', 'task_333174', 'task_342696', 'task_35203

In [ ]:
print(trajectories.keys())

dict_keys([('task_10000', 1), ('task_10000', 2), ('task_10000', 3), ('task_10000', 4), ('task_1001491', 1), ('task_1001491', 2), ('task_1001491', 3), ('task_1001491', 4), ('task_1006812', 1), ('task_1006812', 2), ('task_1006812', 3), ('task_1006812', 4), ('task_100831', 1), ('task_100831', 2), ('task_100831', 3), ('task_100831', 4), ('task_1011056', 1), ('task_1011056', 2), ('task_1011056', 3), ('task_1011056', 4), ('task_1011210', 1), ('task_1011210', 2), ('task_1011210', 3), ('task_1011210', 4), ('task_1013977', 1), ('task_1013977', 2), ('task_1013977', 3), ('task_1013977', 4), ('task_1015088', 1), ('task_1015088', 2), ('task_1015088', 3), ('task_1015088', 4), ('task_1018710', 1), ('task_1018710', 2), ('task_1018710', 3), ('task_1018710', 4), ('task_1025006', 1), ('task_1025006', 2), ('task_1025006', 3), ('task_1025006', 4), ('task_1025939', 1), ('task_1025939', 2), ('task_1025939', 3), ('task_1025939', 4), ('task_1029150', 1), ('task_1029150', 2), ('task_1029150', 3), ('task_1029150

In [ ]:
## optimization loop with loss (write loss in  a seperate function)

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch.optim as optim
import torch.nn.functional as F
import random
import bitsandbytes as bnb
import re
import torch

learning_rate = 2e-5
# Use 8-bit optimizer to save memory
optimizer = bnb.optim.AdamW8bit(policy_model.parameters(), lr=learning_rate, betas=(0.9,0.95), eps=1e-8, weight_decay=0.01)

epochs = 3
k=4
ACCUM_STEPS = 8

# --- RESUME SETTINGS ---
# Just set the path, e.g., './ppo_checkpoint_epoch_1_task_50'. The code handles the rest.
resume_from_checkpoint = None

start_epoch = 0
skip_tasks = 0

if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
    print(f"Loading model weights from {resume_from_checkpoint}...")
    policy_model.lm.load_adapter(resume_from_checkpoint, "default")
    policy_model.value_head.load_state_dict(torch.load(os.path.join(resume_from_checkpoint, 'value_head.pt')))

    # Automatically determine epoch and skip_tasks from folder name
    match = re.search(r"epoch_(\d+)(?:_task_(\d+))?", resume_from_checkpoint)
    if match:
        start_epoch = int(match.group(1)) - 1
        skip_tasks = int(match.group(2)) if match.group(2) else 0
    print(f"Resuming at epoch {start_epoch+1}, skipping first {skip_tasks} tasks.")

print(start_epoch, skip_tasks)
for ppo_epoch in range(start_epoch, epochs):
  print(f'\nTraining for epoch {ppo_epoch+1}')

  # Deterministic shuffle per epoch
  random.seed(42 + ppo_epoch)
  shuffled_ids = task_ids.copy()
  random.shuffle(shuffled_ids)

  # Skip already processed tasks for the resumed epoch
  tasks_to_process = shuffled_ids[skip_tasks:] if ppo_epoch == start_epoch else shuffled_ids
  processed = skip_tasks if ppo_epoch == start_epoch else 0

  optimizer.zero_grad()
  policy_model.train()

  for task_id in tasks_to_process:

    input_text = df[df['task_id'] == task_id]['quiz_text'].to_list()[0]

    if not any((task_id, j + 1) in trajectories for j in range(k)):
      continue

    print(f'Training for epoch: {ppo_epoch+1} and task: {task_id}')

    for i in range(k): # optimizing for each rollout
      if (task_id, i + 1) not in trajectories:
        continue
      torch.cuda.empty_cache()
      initial_prompt_input_ids = generate_input(input_text, tokenizer)
      current_prompt_input_ids = initial_prompt_input_ids

      traj = trajectories[(task_id, i+1)]
      actions, log_prob_olds, advantages, returns  = traj['actions'], traj['log_probs_old'], traj['advantages'], traj['returns']

      # Reshape actions and related tensors from (gen_len, 1) to (1, gen_len)
      actions = actions.transpose(0, 1) # Shape becomes (1, gen_len)
      log_prob_olds = log_prob_olds.transpose(0, 1) # Shape becomes (1, gen_len)
      advantages = advantages.transpose(0, 1) # Shape becomes (1, gen_len)
      returns = returns.transpose(0, 1) # Shape becomes (1, gen_len)

      prompt_len = initial_prompt_input_ids.shape[1]
      gen_len = actions.shape[1] # now actions is (1, gen_len)

      full_sequence = torch.cat(
          [current_prompt_input_ids, actions],
          dim=1)

      logits, value_pred = policy_model(full_sequence, past_key_values=None, use_cache=False)

      # Slice logits aligned to generated tokens
      gen_logits = logits[:, prompt_len-1 : prompt_len+gen_len-1, :].contiguous()

      # Free up full logits immediately
      del logits

      # Calculate log_probs and entropy manually to avoid OOM from Categorical
      log_softmax_gen_probs = F.log_softmax(gen_logits / TEMPERATURE, dim=-1)   # (B, T_gen, V)

      # Free gen_logits as it's no longer needed
      del gen_logits

      log_prob_new = log_softmax_gen_probs.gather(dim=-1, index=actions.unsqueeze(-1)).squeeze(-1)

      probs = torch.exp(log_softmax_gen_probs) # reuse log_softmax to get probs
      entropy = -(probs * log_softmax_gen_probs).sum(dim=-1).mean()

      # Free large probability tensors
      del probs, log_softmax_gen_probs

      ratio = torch.exp(log_prob_new - log_prob_olds)

      with torch.no_grad():
          log_ratio = log_prob_new - log_prob_olds
          approx_kl = (log_ratio.exp() - 1 - log_ratio).mean().item()
          frac_clipped = ((ratio < 0.8) | (ratio > 1.2)).float().mean().item()

      if processed % 20 == 0 and i == 0:
        print(f'epoch={ppo_epoch} task={task_id} kl={approx_kl:.4f} clip_frac={frac_clipped:.3f}')

      # Slice value_pred to match the generated tokens
      # Align values the same way
      value_pred_gen = value_pred[:, prompt_len-1 : prompt_len+gen_len-1]
      value_pred_gen = value_pred_gen.squeeze(-1)   # [1, T_gen]
      del value_pred

      loss = ppo_loss(ratio, advantages, value_pred_gen, returns, entropy)

      # Backpropagate immediately to free the computation graph for this rollout
      scaled_loss = loss / (k * ACCUM_STEPS)

      # Free temporary scalars/tensors before backward
      del loss, ratio, log_prob_new, value_pred_gen, full_sequence, current_prompt_input_ids
      torch.cuda.empty_cache()

      scaled_loss.backward()

    print('\n')

    processed += 1

    if processed % ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        torch.cuda.empty_cache()

    if processed % 50 == 0:
        print(f'Saving checkpoint for epoch {ppo_epoch+1} after {processed} tasks...')
        output_dir = f'/content/drive/MyDrive/ppo_checkpoint_epoch_{ppo_epoch+1}_task_{processed}'
        os.makedirs(output_dir, exist_ok=True)
        policy_model.lm.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        torch.save(policy_model.value_head.state_dict(), os.path.join(output_dir, 'value_head.pt'))
        print(f'Checkpoint saved to {output_dir}\n')

      ####

      # for action, log_prob_old, advantage, return_val in zip(actions, log_prob_olds, advantages, returns):

      #   logits, value_pred, past_key_values = policy_model(current_prompt_input_ids, past_key_values=past_key_values)

      #   print(value_pred, value_pred.shape)
      #   print(return_val, return_val.shape)

      #   probs = F.softmax(logits[:, -1, :], dim=-1)

      #   dist = torch.distributions.Categorical(probs)
      #   entropy = dist.entropy().mean()

      #   log_prob_new = dist.log_prob(action)

      #   ratio = torch.exp(log_prob_new - log_prob_old)

      #   loss+= ppo_loss(ratio, advantage, value_pred, return_val, entropy)

      #   current_prompt_input_ids = action.unsqueeze(0)

        # input_ids = torch.cat([input_ids, action.unsqueeze(0)], dim = 1)

        #Next state st+1 → prompt + token appended
  # print(f'TOTAL_LOSS_ ALL TASK: {total_loss}')
  # optimizer.zero_grad()

  # total_loss = total_loss / (len(tasks) * k)
  # total_loss.backward()
  # optimizer.step()       # <-- MODEL CHANGES HERE

  if processed % ACCUM_STEPS != 0:   # step on the gradients left over from the last tasks of the epoch
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

  print(f'Saving checkpoint for epoch {ppo_epoch+1}...')

  output_dir = f'/content/drive/MyDrive/ppo_checkpoint_epoch_{ppo_epoch+1}'
  os.makedirs(output_dir, exist_ok=True)
  policy_model.lm.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)
  torch.save(policy_model.value_head.state_dict(), os.path.join(output_dir, 'value_head.pt'))
  print(f'Checkpoint saved to {output_dir}\n')


0 0

Training for epoch 1
Training for epoch: 1 and task: task_216665


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


epoch=0 task=task_216665 kl=0.0066 clip_frac=0.034


Training for epoch: 1 and task: task_851495


Training for epoch: 1 and task: task_1070810


Training for epoch: 1 and task: task_853253


Training for epoch: 1 and task: task_87161


Training for epoch: 1 and task: task_107264


Training for epoch: 1 and task: task_940026


Training for epoch: 1 and task: task_20392


Training for epoch: 1 and task: task_879398


Training for epoch: 1 and task: task_411135


Training for epoch: 1 and task: task_81675


Training for epoch: 1 and task: task_524782


Training for epoch: 1 and task: task_1006812


Training for epoch: 1 and task: task_582644


Training for epoch: 1 and task: task_771680


Training for epoch: 1 and task: task_342696


Training for epoch: 1 and task: task_532292


Training for epoch: 1 and task: task_670517


Training for epoch: 1 and task: task_231774


Training for epoch: 1 and task: task_1035282


Training for epoch: 1 and task: task_396175
epoch=0 task=task_396175 kl=0

/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Checkpoint saved to /content/drive/MyDrive/ppo_checkpoint_epoch_1_task_50

Training for epoch: 1 and task: task_131118


Training for epoch: 1 and task: task_282829


Training for epoch: 1 and task: task_363746


Training for epoch: 1 and task: task_671235


Training for epoch: 1 and task: task_352033


Training for epoch: 1 and task: task_319683


Training for epoch: 1 and task: task_676293


Training for epoch: 1 and task: task_565134


Training for epoch: 1 and task: task_535923


Training for epoch: 1 and task: task_1048367


Training for epoch: 1 and task: task_374407
epoch=0 task=task_374407 kl=0.0079 clip_frac=0.066


Training for epoch: 1 and task: task_1055891


Training for epoch: 1 and task: task_740189


Training for epoch: 1 and task: task_559648


Training for epoch: 1 and task: task_867538


Training for epoch: 1 and task: task_333174


Training for epoch: 1 and task: task_60654


Training for epoch: 1 and task: task_383515


Training for epoch: 1 and task: task_148091



In [ ]:
import torch

# The probabilities should sum to 1 and be non-negative.
# Original list: [0, 2, 0.5, 0.3] - sum is 2.8, also has a value > 1
# Let's create a valid probability distribution for demonstration.
# For example, [0.2, 0.3, 0.4, 0.1] sums to 1.0

probs_list = [0.2, 0.3, 0.4, 0.1]
dist = torch.distributions.Categorical(torch.tensor(probs_list))

action = dist.sample()
log_prob = dist.log_prob(action)

print(f"Action: {action}")
print(f"Log Probability: {log_prob}")

Action: 2
Log Probability: -0.9162907004356384


In [ ]:
# compute returns/rewards


In [ ]:
import json, re, torch

# Define device globally for use in conditional_reward_system
device = "cuda" if torch.cuda.is_available() else "cpu"


# ---- Inputs verbatim from your training log (Zoey/Audrey, rollout 2) ----
name_role_subanswers_gt = [
    '[{"name": "Zoey", "role": "devil"}]',
    '[{"name": "Zoey", "role": "devil"}]',
    '[{"name": "Audrey", "role": "angel"}]',
    '[{"name": "Audrey", "role": "angel"}]',
    '[{"name": "Zoey", "role": "devil"}, {"name": "Audrey", "role": "angel"}]',
]
gt_final_pair            = name_role_subanswers_gt
final_answer             = "Zoey is a devil and Audrey is an angel."
generated_subanswers     = [
    "...<sub_ans>Zoey: devil</sub_ans>",
    "...<sub_ans>Audrey: angel</sub_ans>",
    "Thus, Zoey is a devil and Audrey is an angel.",
]
name_role_subanswers_gen = ["Zoey: devil", "Audrey: angel"]
final_generated_answer   = '{"Zoey": "devil", "Audrey": "angel"}'
sub_ans_token_positions  = [61, 126]
final_ans_index          = 186
T, gamma                 = 189, 0.9

# Define dummy i for the print statement inside conditional_reward_system
i = 1

# ---- Run YOUR notebook's conditional_reward_system on this ----
r = conditional_reward_system(
    gt_clean_subanswers=[],
    generated_subanswers=generated_subanswers,
    name_role_subanswers_gen=name_role_subanswers_gen,
    name_role_subanswers_gt=name_role_subanswers_gt,
    final_generated_answer=final_generated_answer,
    final_answer=final_answer,
    gt_final_pair=gt_final_pair,
    sub_ans_token_positions=sub_ans_token_positions,
    final_ans_index=final_ans_index,
    Acc_curr=None, Acc_prev=None, mode="no_accuracy_gate",
    T=T, gamma=gamma,
)
print("rewards.sum() :", r.sum().item())
print("rewards.mean():", r.mean().item())
print("nonzero pos   :", (r != 0).nonzero().flatten().tolist())
print("nonzero val   :", [round(v.item(), 4) for v in r[r != 0]])

rewards for the task in rollout 2:  tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0., 

In [ ]:
# advantage estimation

# A= gt - value



In [ ]:
inp=tokenizer("what is 3+4 ?", return_tensors="pt")

logits, value_pred = policy_model(inp['input_ids'].to(device), past_key_values=None, use_cache=False)

outputs = torch.argmax(logits, dim=-1)
print(outputs.shape, len(inp['input_ids'][0]))
tokenizer.decode(outputs[0])

torch.Size([1, 7]) 7


' is the103\n The'

In [ ]:
def generate_response(prompt_ids, model, tokenizer, max_new_tokens=256):
    model.eval()

    generated_ids = []
    past_key_values = None
    current_ids = prompt_ids.to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, value, past_key_values = model(current_ids, past_key_values=past_key_values, use_cache=True)

            # Greedy decoding
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

            if next_token_id.item() == tokenizer.eos_token_id:
                break

            generated_ids.append(next_token_id.item())
            current_ids = next_token_id

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# Test with a structured logic puzzle prompt from the dataset
sample_puzzle = final_df.iloc[0]['quiz_text']
# Use the template function we created in cell 70CZCSJmdw5_
formatted_input_ids = generate_input(sample_puzzle, tokenizer)

print(f"Testing puzzle: {sample_puzzle[:100]}...")
response = generate_response(formatted_input_ids, policy_model, tokenizer)
print("\n--- Generated Response ---")
print(response)

Testing puzzle: A very special island is inhabited only by guardians and deceivers. Guardians always tell the truth,...

--- Generated Response ---
 <think>Let's start by considering Elijah's statement. Elijah says "Scarlett is a deceiver". If Elijah is a guardian, he tells the truth, so Scarlett must be a deceiver. If Elijah is a deceiver, he lies, so Scarlett must be a guardian. Ella's statement is "Scarlett is a deceiver and Ella is a deceiver". If Ella is a guardian, both parts of her statement must be true, so Scarlett would be a deceiver and Ella would be a deceiver, which is a contradiction. Therefore, Ella must be a deceiver.</think>
<answer>Ella is a deceiver
<sub_ans>Ella: deceiver</sub_ans></answer>
<think>Since Ella is a deceiver, her statement "Scarlett is a deceiver and Ella is a deceiver" is false. This means Scarlett is not a deceiver. If Scarlett is a guardian, her statement that Elijah is a deceiver would be true, which contradicts the fact that a guardian always tell

In [ ]:
import os

final_save_path = "./final_ppo_model"
os.makedirs(final_save_path, exist_ok=True)

print(f"Saving final model adapters to {final_save_path}...")

# Safely save only the trained LoRA adapter weights and tokenizer
policy_model.lm.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print("Final model saved successfully! You can load it later using PeftModel.from_pretrained()")


Saving final model adapters to ./final_ppo_model...
Final model saved successfully! You can load it later using PeftModel.from_pretrained()


In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define source and destination paths
source_dir = './final_ppo_model'
drive_dir = '/content/drive/MyDrive/final_ppo_model'

print(f"Copying model from {source_dir} to {drive_dir}...")
# Copy the directory to Drive
shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
print("Model successfully copied to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying model from ./final_ppo_model to /content/drive/MyDrive/final_ppo_model...
Model successfully copied to Google Drive!


### Loading the model from Google Drive
In a fresh session, you would first load the base model (`Qwen/Qwen2.5-7B-Instruct`), and then apply the LoRA adapters from your Drive using `PeftModel`. Here is how you can load it back:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer

drive_dir = '/content/drive/MyDrive/final_ppo_model'

print("Loading tokenizer from Drive...")
loaded_tokenizer = AutoTokenizer.from_pretrained(drive_dir)

print("Loading LoRA adapters from Drive...")
# In a new session, you'd load the base_model first.
# Since we already have the base model in memory as `model.get_base_model()`, we can load the adapters onto it:
loaded_model = PeftModel.from_pretrained(model.get_base_model(), drive_dir)

print("Successfully loaded the model from Google Drive!")

Loading tokenizer from Drive...
Loading LoRA adapters from Drive...


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Successfully loaded the model from Google Drive!


In [ ]:
# Pick a puzzle from index 30 (which was not in our top 20 training slice)
external_sample = df.groupby(['task_id', 'quiz_text']).first().reset_index().iloc[30]
external_puzzle = external_sample['quiz_text']

print(f"Testing on an UNSEEN puzzle (Task ID: {external_sample['task_id']}):")
print(f"{external_puzzle[:150]}...\n")

# Format and generate
formatted_external_ids = generate_input(external_puzzle, tokenizer)
external_response = generate_response(formatted_external_ids, policy_model, tokenizer, max_new_tokens=400)

print("--- Generated Response for Unseen Puzzle ---")
print(external_response)

Testing on an UNSEEN puzzle (Task ID: task_142859):
A very special island is inhabited only by pioneers and laggards. Pioneers always tell the truth, and laggards always lie. You meet 2 inhabitants: Ava...

--- Generated Response for Unseen Puzzle ---
 <think>Ava says "Ava is a pioneer". If Ava is a pioneer, then this statement is true. If Ava is a laggard, then this statement is false. Since a laggard always lies, Ava must be a pioneer.</think>
<answer>Ava is a pioneer
<sub_ans>Ava: pioneer</sub_ans></answer>
<think>David says "If Ava is a pioneer then David is a pioneer if and only if Ava is a pioneer if and only if Ava is a laggard". Since Ava is a pioneer, the statement "Ava is a pioneer if and only if Ava is a laggard" is false. A pioneer never says something false, so David must be a laggard.</think>
<answer>David is a laggard
<sub_ans>David: laggard</sub_ans></answer>
<think>So, the final answer is:</think>
<answer>Ava is a pioneer, and David is a laggard.
<final_answer>{"Ava": 

In [ ]:
# To see if RL helped, let's run the same puzzle through the Reference (Un-trained) model
print("--- Output from REFERENCE MODEL (Pre-RL) ---")

# We don't need a separate ref_model copy.
# We can just temporarily disable the LoRA adapters on the trained policy_model!
policy_model.eval()

with policy_model.lm.disable_adapter():
    ref_response = generate_response(formatted_external_ids, policy_model, tokenizer, max_new_tokens=400)
print(ref_response)

print("\n\n--- Output from POLICY MODEL (After RL) ---")
print(external_response)


--- Output from REFERENCE MODEL (Pre-RL) ---
 <think>Ava says "Ava is a pioneer". If Ava is a laggard, she would be lying, which means the statement "Ava is a pioneer" would be false. This is a contradiction, so Ava must be a pioneer.</think>
<answer>Ava is a pioneer
<sub_ans>Ava: pioneer</sub_ans></answer>
<think>Ava is a pioneer, so her statement "Ava is a pioneer" is true. David's statement is "If Ava is a pioneer then David is a pioneer if and only if Ava is a pioneer if and only if Ava is a laggard". Since Ava is a pioneer, the first part "If Ava is a pioneer" is true. For the entire statement to be true, the second part "David is a pioneer if and only if Ava is a pioneer if and only if Ava is a laggard" must also be true. Since Ava is a pioneer, "Ava is a pioneer if and only if Ava is a laggard" is false (a pioneer cannot be a laggard). Therefore, David must be a laggard to make the entire statement false.</think>
<answer>David is a laggard
<sub_ans>David: laggard</sub_ans></answ

In [ ]:
"""
TEST 2 — Does Ella's answer actually depend on Mia's role?

Logic:
  Mia's statement is a tautology -> Mia: angel.
  Ella says "Mia is a devil is not true" (= "Mia is an angel").
  So Ella's role is DOWNSTREAM of Mia's:
      Mia angel  -> Ella's claim is TRUE  -> Ella: angel
      Mia devil  -> Ella's claim is FALSE -> Ella: devil

We feed the model the SAME prefix twice, changing ONLY Mia's recorded role:
  CONTROL   : "<answer>Mia: angel</answer>"   (truth)
  CORRUPTED : "<answer>Mia: devil</answer>"   (a lie we inject)

Then we let the model generate Ella's step and read which role it picks.

  - If Ella flips angel->devil when we lie about Mia  -> model is USING the
    Mia fact (reasoning, just not printing it).
  - If Ella stays angel regardless                    -> model ignores the
    premise (pattern-match / shortcut).

NOTE: edit MODEL_PATH to your checkpoint (base + LoRA adapter, or merged).
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct"   # <-- change to your trained checkpoint
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------
# 1) The shared prefix: the puzzle + Mia's step, stopping right before Ella.
#    Everything here is IDENTICAL across conditions except Mia's role token.
#    {mia_role} is the only thing we swap.
# ----------------------------------------------------------------------
PREFIX_TEMPLATE = (
    "You are solving a Knights-and-Knaves puzzle. Angels always tell the truth; "
    "devils always lie.\n"
    "Mia says: \"Mia is a devil and Mia is a devil or Mia is an angel or Mia is an angel.\"\n"
    "Ella says: \"Mia is a devil is not true.\"\n"
    "Reason step by step, giving each role inside <answer>Name: role</answer>.\n"
    "<think>Mia's statement is true.</think>\n"
    "<answer>Mia: {mia_role}</answer>\n"
    "<think>"          # we stop here; the model continues Ella's reasoning + answer
)

CONTROL_ROLE   = "angel"   # the truth
CORRUPTED_ROLE = "devil"   # the lie we inject

def build_prompt(mia_role):
    return PREFIX_TEMPLATE.format(mia_role=mia_role)

# ----------------------------------------------------------------------
# 2) Generate Ella's continuation and pull the role out of the first
#    <answer>Ella: ...</answer> the model writes.
# ----------------------------------------------------------------------
import re
def extract_ella_role(text):
    m = re.search(r"Ella:\s*(angel|devil)", text, re.IGNORECASE)
    return m.group(1).lower() if m else "NOT_FOUND"

def run(model, tok, mia_role):
    prompt = build_prompt(mia_role)
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,        # greedy / temp 0 -> deterministic, clean test
            pad_token_id=tok.eos_token_id,
        )
    full = tok.decode(out[0], skip_special_tokens=True)
    continuation = full[len(prompt):]   # only what the model added
    return extract_ella_role(continuation), continuation.strip()

# ----------------------------------------------------------------------
# 3) Run both conditions and compare. This block is the entire experiment.
# ----------------------------------------------------------------------
def main():
    print("Loading model... (edit MODEL_PATH if this is the base model)")
    tok = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

    ctrl_role, ctrl_text = run(model, tok, CONTROL_ROLE)
    corr_role, corr_text = run(model, tok, CORRUPTED_ROLE)

    print("\n================ CONTROL  (Mia: angel, the truth) ================")
    print("Ella continuation:", ctrl_text)
    print(">>> Ella role:", ctrl_role)

    print("\n================ CORRUPTED (Mia: devil, injected lie) ===========")
    print("Ella continuation:", corr_text)
    print(">>> Ella role:", corr_role)

    print("\n======================== VERDICT ================================")
    if ctrl_role == "NOT_FOUND" or corr_role == "NOT_FOUND":
        print("INCONCLUSIVE: couldn't parse an Ella role in one condition.")
        print("Inspect the continuations above; adjust max_new_tokens or the regex.")
    elif ctrl_role != corr_role:
        print(f"FLIP  ({ctrl_role} -> {corr_role}).")
        print("The lie propagated. The model USES Mia's role to decide Ella's.")
        print("=> The skipped reasoning is load-bearing (latent, just unspoken).")
    else:
        print(f"NO FLIP (stayed {ctrl_role} in both).")
        print("The lie was ignored. Ella's answer is decoupled from the premise.")
        print("=> Looks like pattern-match / shortcut, not premise-driven reasoning.")

if __name__ == "__main__":
    main()

In [ ]:
"""
TEST 2 (v3) — better PROMPT. The model kept fumbling the logic because the
old prompt never spelled out HOW to go from a fact to a role. Here we state
the exact 3-step rule, so a wrong answer means "used the premise wrong" is
no longer just sloppiness from an under-specified task.

Still the lie-test:
  CONTROL  : tell it Mia is an angel   (truth)  -> correct Ella = angel
  CORRUPT  : tell it Mia is a devil    (lie)    -> correct Ella = devil
If Ella flips, the answer DEPENDS on the Mia premise (model reasons from it).
"""
import re, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          StoppingCriteria, StoppingCriteriaList)

MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct"   # <-- your checkpoint
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# The fix is ALL in this prompt: we give the rule, a worked mini-example on a
# DIFFERENT name (so we don't leak the answer), and the exact step order.
PROMPT_TEMPLATE = (
    "Knights-and-Knaves logic.\n"
    "Rules:\n"
    "  - An angel always makes TRUE statements.\n"
    "  - A devil always makes FALSE statements.\n"
    "  - To find someone's role: (1) work out whether their statement is TRUE "
    "or FALSE using the known facts, then (2) if their statement is TRUE they "
    "are an angel; if FALSE they are a devil.\n"
    "  - Note: whether a person lies is SEPARATE from whether a sentence about "
    "them is true. 'X is a devil' is TRUE when X really is a devil.\n\n"
    "Worked example (different puzzle):\n"
    "  Fact: Tom is an angel. Sara says: \"Tom is an angel.\"\n"
    "  Step 1: 'Tom is an angel' is TRUE (Tom really is an angel).\n"
    "  Step 2: Sara's statement is TRUE, so Sara is an angel.\n\n"
    "Now your puzzle:\n"
    "  Fact: Mia is a {mia_role}.\n"
    "  Ella says: \"Mia is a devil is not true.\"\n"
    "  Step 1: decide if Ella's statement is TRUE or FALSE using the fact.\n"
    "  Step 2: give Ella's role.\n"
    "Answer as <answer>Ella: angel</answer> or <answer>Ella: devil</answer>.\n"
    "<think>"
)

class StopOnAnswerClose(StoppingCriteria):
    def __init__(self, tok, plen): self.tok, self.plen = tok, plen
    def __call__(self, ids, scores, **kw):
        return "</answer>" in self.tok.decode(ids[0][self.plen:], skip_special_tokens=True)

def first_role(t):
    m = re.search(r"Ella\w*\s*:?\s*(angel|devil)", t, re.IGNORECASE)
    return m.group(1).lower() if m else "NOT_FOUND"

def run(model, tok, mia_role):
    p = PROMPT_TEMPLATE.format(mia_role=mia_role)
    inp = tok(p, return_tensors="pt").to(DEVICE)
    stop = StoppingCriteriaList([StopOnAnswerClose(tok, inp["input_ids"].shape[1])])
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=250, do_sample=False,
                             pad_token_id=tok.eos_token_id, stopping_criteria=stop)
    cont = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return first_role(cont), cont.strip()

def main():
    tok = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16,
                                                 device_map="auto").eval()
    ctrl, ct = run(model, tok, "angel")   # truth -> Ella should be angel
    corr, rt = run(model, tok, "devil")   # lie   -> Ella should be devil
    print("\n== CONTROL (Mia: angel) ==\n", ct, "\n>>> Ella:", ctrl)
    print("\n== CORRUPTED (Mia: devil) ==\n", rt, "\n>>> Ella:", corr)
    print("\n== VERDICT ==")
    if "NOT_FOUND" in (ctrl, corr):
        print("INCONCLUSIVE: couldn't parse a role. Read continuations above.")
    elif ctrl == "angel" and corr == "devil":
        print("CLEAN FLIP (angel->devil). Ella's answer tracks the Mia premise "
              "AND tracks it CORRECTLY. The model reasons from the premise.")
    elif ctrl != corr:
        print(f"FLIP but odd ({ctrl}->{corr}); read the reasoning.")
    else:
        print(f"NO FLIP (stayed {ctrl}). Read BOTH <think> blocks: did the "
              "corrupted run use the premise and mis-reason, or ignore it?")

if __name__ == "__main__":
    main()

In [ ]:
from google.colab import runtime
runtime.unassign()